# NDVI from Sentinel-2 — Tel Aviv

Calculates NDVI for Tel Aviv using the most recent cloud-free Sentinel-2 SR imagery via Google Earth Engine.

**Output:** `tel_aviv_ndvi.tif` exported to your Google Drive (`GEE_Exports/` folder).

**Run in:** Google Colab

In [ ]:
# Install dependencies (Colab only)
!pip install geemap -q

In [ ]:
import ee
import geemap

# Authenticate and initialise — opens a browser login on first run
ee.Authenticate()
ee.Initialize(project='your-gee-project-id')  # <-- replace with your GEE project ID

In [ ]:
# ── Study area ──────────────────────────────────────────────────────────────
# Tel Aviv municipal bounding box (WGS84)
tel_aviv = ee.Geometry.BBox(34.739, 32.030, 34.840, 32.130)

# ── Date range: last 6 months → pick best available ────────────────────────
import datetime
end_date   = datetime.date.today().isoformat()
start_date = (datetime.date.today() - datetime.timedelta(days=180)).isoformat()
print(f'Searching imagery from {start_date} to {end_date}')

In [ ]:
# ── Load Sentinel-2 Surface Reflectance collection ──────────────────────────
def mask_s2_clouds(image):
    """Mask clouds using the SCL (Scene Classification Layer) band."""
    scl = image.select('SCL')
    # Keep: 4=vegetation, 5=bare soil, 6=water, 7=unclassified, 11=snow/ice
    # Remove: 3=cloud shadow, 8=cloud medium, 9=cloud high, 10=cirrus
    cloud_mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return image.updateMask(cloud_mask)

def add_ndvi(image):
    """Add NDVI band: (B8 - B4) / (B8 + B4)."""
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)

collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(tel_aviv)
    .filterDate(start_date, end_date)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 15))
    .map(mask_s2_clouds)
    .map(add_ndvi)
)

print(f'Images found: {collection.size().getInfo()}')

In [ ]:
# ── Pick most recent single image ───────────────────────────────────────────
most_recent = collection.sort('system:time_start', False).first()

date_str = ee.Date(most_recent.get('system:time_start')).format('YYYY-MM-dd').getInfo()
cloud_pct = most_recent.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()
print(f'Most recent image date : {date_str}')
print(f'Cloud cover            : {cloud_pct:.1f}%')

ndvi = most_recent.select('NDVI').clip(tel_aviv)

In [ ]:
# ── Interactive map preview ──────────────────────────────────────────────────
Map = geemap.Map(center=[32.08, 34.79], zoom=12)

ndvi_vis = {
    'min': -0.2,
    'max': 0.6,
    'palette': ['#d73027', '#f46d43', '#fdae61', '#fee08b',
                '#d9ef8b', '#a6d96a', '#66bd63', '#1a9850']
}

Map.addLayer(ndvi, ndvi_vis, f'NDVI {date_str}')
Map.add_colorbar(ndvi_vis, label='NDVI', layer_name=f'NDVI {date_str}')
Map

In [ ]:
# ── Export to Google Drive ───────────────────────────────────────────────────
# Resolution: 10m (native Sentinel-2 resolution for B4/B8)
export_task = ee.batch.Export.image.toDrive(
    image=ndvi,
    description=f'tel_aviv_ndvi_{date_str}',
    folder='GEE_Exports',
    fileNamePrefix=f'tel_aviv_ndvi_{date_str}',
    region=tel_aviv,
    scale=10,
    crs='EPSG:4326',
    maxPixels=1e9,
    fileFormat='GeoTIFF'
)

export_task.start()
print('Export started — check the Tasks tab in code.earthengine.google.com')
print(f'File will appear in Google Drive → GEE_Exports/tel_aviv_ndvi_{date_str}.tif')

In [ ]:
# ── (Optional) Monitor export status ────────────────────────────────────────
import time

while export_task.active():
    status = export_task.status()
    print(f"State: {status['state']}  |  {datetime.datetime.now().strftime('%H:%M:%S')}")
    time.sleep(30)

print(f"Final state: {export_task.status()['state']}")

## After the Export Finishes

1. Download `tel_aviv_ndvi_<date>.tif` from Google Drive.
2. Place it in `Layers/` in this project.
3. Load it in subsequent notebooks with:

```python
import rasterio
import numpy as np

with rasterio.open('Layers/tel_aviv_ndvi_<date>.tif') as src:
    ndvi_array = src.read(1)   # shape: (rows, cols)
    transform  = src.transform
    crs        = src.crs
```

Or as an `xarray` dataset:
```python
import rioxarray
ndvi_xr = rioxarray.open_rasterio('Layers/tel_aviv_ndvi_<date>.tif')
```